In [1]:
import cv2
import json
import os
import importlib

from similarity_check import calculate_similarity
from similarity_check import extract_similarity_features
from similarity_check import calculate_similarity_from_features
from embedder import DINOv2Embedder
from sklearn.metrics.pairwise import cosine_similarity

c:\Computer Vision Project\Supermarket shelves\Supermarket shelves\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedder = DINOv2Embedder()

Loading facebook/dinov2-base on cpu...


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2021.73it/s]

Model loaded successfully.


In [16]:


from pathlib import Path

BASE_DIR = Path().resolve()
 
IMAGE_NAME = "025"
 
IMAGE_PATH = os.path.join(BASE_DIR, "images", IMAGE_NAME + ".jpg")
ANNOTATION_PATH = os.path.join(BASE_DIR, "annotations", IMAGE_NAME + ".jpg.json")
 
OUTPUT_PATH = os.path.join(BASE_DIR, "results", IMAGE_NAME + "_misplaced_embedding.jpg")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
 

In [13]:
# Ignore tiny boxes
MIN_BOX_AREA = 20000
 
# Products whose y-centers differ by less than this
# are assumed to belong to the same shelf row.
ROW_THRESHOLD = 80
 
SIMILARITY_THRESHOLD = 0.65

DISTANCE_THRESHOLD = 200  # Maximum distance between products to consider them neighbors
 

## Rough Work

In [71]:
image = cv2.imread(IMAGE_PATH)
 
with open(ANNOTATION_PATH) as f:
    data = json.load(f)
 
 

In [6]:
data

{'description': '',
 'tags': [],
 'size': {'height': 4608, 'width': 3456},
 'objects': [{'id': 961442428,
   'classId': 10213294,
   'description': '',
   'geometryType': 'rectangle',
   'labelerLogin': 'humanintheloop',
   'createdAt': '2022-07-05T10:52:19.278Z',
   'updatedAt': '2022-07-05T10:52:19.278Z',
   'tags': [],
   'classTitle': 'Product',
   'points': {'exterior': [[3346, 16], [3454, 443]], 'interior': []}},
  {'id': 961442431,
   'classId': 10213294,
   'description': '',
   'geometryType': 'rectangle',
   'labelerLogin': 'humanintheloop',
   'createdAt': '2022-07-05T10:52:19.278Z',
   'updatedAt': '2022-07-05T10:52:19.278Z',
   'tags': [],
   'classTitle': 'Product',
   'points': {'exterior': [[3188, 32], [3339, 444]], 'interior': []}},
  {'id': 961442435,
   'classId': 10213294,
   'description': '',
   'geometryType': 'rectangle',
   'labelerLogin': 'humanintheloop',
   'createdAt': '2022-07-05T10:52:19.278Z',
   'updatedAt': '2022-07-05T10:52:19.278Z',
   'tags': [],
  

In [8]:
products = []
 
for obj in data["objects"]:
 
    if obj["classTitle"] != "Product":
        continue
 
    if obj["geometryType"] != "rectangle":
        continue
 
    (x1, y1), (x2, y2) = obj["points"]["exterior"]
 
    x1 = int(x1)
    y1 = int(y1)
    x2 = int(x2)
    y2 = int(y2)
 
    area = (x2 - x1) * (y2 - y1)
 
    if area < MIN_BOX_AREA:
        continue
    print(f"Processing Object {obj['id']} with bbox: ({x1}, {y1}, {x2}, {y2}) and area: {area}")
    crop = image[y1:y2, x1:x2]
 
    products.append({
        "bbox": (x1, y1, x2, y2),
        "crop": crop,
        "cx": (x1 + x2) / 2,
        "cy": (y1 + y2) / 2,
        "status": "Correct"
    })

Processing Object 961442428 with bbox: (3346, 16, 3454, 443) and area: 46116
Processing Object 961442431 with bbox: (3188, 32, 3339, 444) and area: 62212
Processing Object 961442435 with bbox: (3025, 24, 3181, 438) and area: 64584
Processing Object 961442440 with bbox: (2857, 19, 3018, 431) and area: 66332
Processing Object 961442443 with bbox: (2693, 16, 2848, 424) and area: 63240
Processing Object 961442456 with bbox: (2519, 18, 2686, 426) and area: 68136
Processing Object 961442461 with bbox: (2351, 20, 2515, 426) and area: 66584
Processing Object 961442470 with bbox: (2177, 25, 2348, 428) and area: 68913
Processing Object 961442549 with bbox: (2019, 47, 2174, 450) and area: 62465
Processing Object 961442558 with bbox: (1844, 19, 2013, 426) and area: 68783
Processing Object 961442573 with bbox: (1664, 8, 1837, 418) and area: 70930
Processing Object 961442577 with bbox: (1496, 7, 1659, 420) and area: 67319
Processing Object 961442578 with bbox: (1321, 5, 1491, 415) and area: 69700
Pr

In [25]:
products.sort(key=lambda p: p["cy"])
 
rows = []
 
for product in products:
    print(f"Processing Product {product['cx']}")
    assigned = False
 
    for row in rows:
 
        if abs(product["cy"] - row[0]["cy"]) < 200:
            row.append(product)
            assigned = True
            break
            
    if not assigned:
        rows.append([product])

Processing Product 884.5
Processing Product 541.0
Processing Product 1061.5
Processing Product 715.0
Processing Product 1406.0
Processing Product 365.0
Processing Product 190.5
Processing Product 50.0
Processing Product 1750.5
Processing Product 1234.5
Processing Product 1577.5
Processing Product 2770.5
Processing Product 2602.5
Processing Product 1928.5
Processing Product 2433.0
Processing Product 2937.5
Processing Product 2262.5
Processing Product 3400.0
Processing Product 3103.0
Processing Product 3263.5
Processing Product 2096.5
Processing Product 886.0
Processing Product 710.5
Processing Product 1061.0
Processing Product 535.0
Processing Product 359.5
Processing Product 1410.0
Processing Product 1581.5
Processing Product 1235.5
Processing Product 46.0
Processing Product 1752.5
Processing Product 2265.5
Processing Product 2609.0
Processing Product 2435.5
Processing Product 183.5
Processing Product 1933.0
Processing Product 2946.0
Processing Product 2778.0
Processing Product 3116.0


In [22]:
print(len(rows))

9


## Real Code

In [5]:
import json
import numpy as np

with open("class_embeddings.json", "r") as f:
    class_embeddings = json.load(f)

# Convert embeddings and features to NumPy arrays
for class_name in class_embeddings:
    for image_name in class_embeddings[class_name]:

        sample = class_embeddings[class_name][image_name]

        # Embedding
        sample["embedding"] = np.array(sample["embedding"])

        # Histogram
        sample["features"]["histogram"] = np.array(
            sample["features"]["histogram"],
            dtype=np.float32
        )

        # ORB descriptors
        if sample["features"]["descriptors"] is not None:
            sample["features"]["descriptors"] = np.array(
                sample["features"]["descriptors"],
                dtype=np.uint8
            )

In [17]:
# ============================================================
# Load
# ============================================================
 
image = cv2.imread(IMAGE_PATH)
 
with open(ANNOTATION_PATH) as f:
    data = json.load(f)
 
 
# ============================================================
# Extract products only
# ============================================================
 
products = []
 
for obj in data["objects"]:
 
    if obj["classTitle"] != "Product":
        continue
 
    if obj["geometryType"] != "rectangle":
        continue
 
    (x1, y1), (x2, y2) = obj["points"]["exterior"]
 
    x1 = int(x1)
    y1 = int(y1)
    x2 = int(x2)
    y2 = int(y2)
 
    area = (x2 - x1) * (y2 - y1)
 
    if area < MIN_BOX_AREA:
        continue
 
    crop = image[y1:y2, x1:x2]
 
    products.append({
        "bbox": (x1, y1, x2, y2),
        "crop": crop,
        "cx": (x1 + x2) / 2,
        "cy": (y1 + y2) / 2,
        "status": "Correct"
    })
 
 
# ============================================================
# Group into shelf rows
# ============================================================
misplaced_products = []

products.sort(key=lambda p: p["bbox"][3])
 
rows = []
 
for product in products:
 
    assigned = False
 
    for row in rows:
 
        if abs(product["bbox"][3] - row[0]["bbox"][3]) < ROW_THRESHOLD:
            row.append(product)
            assigned = True
            break
            
    if not assigned:
        rows.append([product])
 
 
# ============================================================
# Sort each row from left to right
# ============================================================
 
for row in rows:
    row.sort(key=lambda p: p["cx"])
 

#Setting a gap threshold to determine if two products are neighbors. If the gap between two products is less than this threshold, they are considered neighbors.
#We will consider threshold as average gap between products in a row. If the gap between two products is less than this threshold, they are considered neighbors.

average_gap = 0
row_count = 0
for row in rows:
    row_count = row_count+1
    print(f"{len(row)} products in row {row_count} ")
    for i in range(len(row) - 1):
        gap = row[i + 1]["bbox"][0] - row[i]["bbox"][2]
        average_gap += gap
average_gap /= max(sum(len(row) - 1 for row in rows), 1)
print(f"Average gap between products: {average_gap}")

# ============================================================
# Compare neighbours
# ============================================================
for row_index, row in enumerate(rows):
 
    embeddings = []
 
    for product in row:

        emb = embedder.get_embedding(product["crop"])
        embeddings.append(emb)
    for i in range(len(row)):
 
        left_similarity = None
        right_similarity = None
 
        if i > 0:
            # left_distance = abs(row[i]["cx"] - row[i - 1]["cx"])
            left_gap = row[i]["bbox"][0] - row[i - 1]["bbox"][2]
            
            # print(f"Left gap between product {i} and {i-1}: {left_gap}")
            if left_gap <= average_gap:
                pass
                left_similarity_from_embeddings = embedder.similarity_from_embeddings(
                    embeddings[i],
                    embeddings[i - 1]
                )
                left_similarity_from_features = calculate_similarity(row[i]["crop"], row[i - 1]["crop"])

                left_similarity = (left_similarity_from_embeddings * 0.4 + left_similarity_from_features * 0.6)
                # left_similarity = calculate_similarity(row[i]["crop"], row[i - 1]["crop"])
 
        if i < len(row) - 1:
            # right_distance = abs(row[i]["cx"] - row[i + 1]["cx"])
            right_gap = row[i + 1]["bbox"][0] - row[i]["bbox"][2]
            # print(f"Right gap between product {i} and {i+1}: {right_gap}")
            if right_gap <= average_gap:
                
                right_similarity_from_features = calculate_similarity(row[i]["crop"], row[i + 1]["crop"])
                right_similarity_from_embeddings = embedder.similarity_from_embeddings(
                    embeddings[i],
                    embeddings[i + 1]
                )
                right_similarity = (right_similarity_from_embeddings * 0.4 + right_similarity_from_features * 0.6) 
                # right_similarity = calculate_similarity(row[i]["crop"], row[i + 1]["crop"])
 
        neighbours = []
 
        if left_similarity is not None:
            neighbours.append(left_similarity)
 
        if right_similarity is not None:
            neighbours.append(right_similarity)
 
        if len(neighbours) == 0:
            continue
 
        if max(neighbours) < SIMILARITY_THRESHOLD:
            row[i]["status"] = "Misplaced"

            best_class = None
            best_average = -1

            for class_name, samples in class_embeddings.items():

                similarities = []

                features = extract_similarity_features(row[i]["crop"])

                for sample in samples.values():

                    embedding_similarity = embedder.similarity_from_embeddings(
                        embeddings[i],
                        sample["embedding"]
                    )

                    feature_similarity = calculate_similarity_from_features(
                        features,
                        sample["features"]
                    )

                    # However you want to combine them
                    similarities.append((embedding_similarity + feature_similarity) / 2)

                average_similarity = sum(similarities) / len(similarities)
                print(f"Product {i+1} in row {row_index + 1}: Average similarity to class '{class_name}' = {average_similarity:.4f}")

                if average_similarity > best_average:
                    best_average = average_similarity
                    best_class = class_name

            misplaced_product = {
                "class": best_class,
                "bbox": row[i]["bbox"],
                "row": row_index + 1,
                "product": i + 1
            }

            misplaced_products.append(misplaced_product)
            
    

21 products in row 1 
21 products in row 2 
19 products in row 3 
21 products in row 4 
21 products in row 5 
21 products in row 6 
16 products in row 7 
5 products in row 8 
15 products in row 9 
5 products in row 10 
16 products in row 11 
Average gap between products: 11.08235294117647
Product 11 in row 2: Average similarity to class 'class advil' = 0.0778
Product 11 in row 2: Average similarity to class 'class coke' = 0.3994
Product 11 in row 2: Average similarity to class 'class coloxyl' = 0.1353
Product 11 in row 2: Average similarity to class 'class dietcoke' = 0.3316
Product 11 in row 2: Average similarity to class 'class dimetapp' = 0.0825
Product 11 in row 2: Average similarity to class 'class dymadon' = 0.0534
Product 11 in row 2: Average similarity to class 'class fanta' = 0.6556
Product 11 in row 2: Average similarity to class 'class glycerol' = 0.0730
Product 11 in row 2: Average similarity to class 'class littlecoughs' = 0.0840
Product 11 in row 2: Average similarity to 

In [18]:
# ============================================================
# Draw results
# ============================================================
 
output = image.copy()
 
for product in products:
 
    x1, y1, x2, y2 = product["bbox"]
 
    if product["status"] == "Misplaced":
 
        color = (0, 0, 255)
        label = "Misplaced"
 
    else:
 
        color = (0, 255, 0)
        label = "Correctly Placed"
 
    cv2.rectangle(
        output,
        (x1, y1),
        (x2, y2),
        color,
        3
    )
 
    cv2.putText(
        output,
        label,
        (x1, y1 - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        color,
        2
    )
 
cv2.imwrite(OUTPUT_PATH, output)
 
print("Saved:", OUTPUT_PATH)

Saved: C:\Computer Vision Project\Supermarket shelves\Supermarket shelves\Supermarket shelves\results\025_misplaced_embedding.jpg


In [19]:
for misplaced_product in misplaced_products:
    print(f"Misplaced Product: Class={misplaced_product['class']}, Row={misplaced_product['row']}, Product={misplaced_product['product']}, BBox={misplaced_product['bbox']}")

Misplaced Product: Class=class fanta, Row=2, Product=11, BBox=(1667, 428, 1838, 844)
Misplaced Product: Class=class coke, Row=3, Product=1, BBox=(368, 1060, 548, 1489)
Misplaced Product: Class=class coke, Row=3, Product=2, BBox=(553, 1112, 641, 1519)
Misplaced Product: Class=class dietcoke, Row=3, Product=3, BBox=(646, 1059, 813, 1487)
Misplaced Product: Class=class coke, Row=5, Product=12, BBox=(1895, 2122, 2045, 2534)
Misplaced Product: Class=class coke, Row=5, Product=13, BBox=(2050, 2111, 2213, 2528)
Misplaced Product: Class=class fanta, Row=5, Product=14, BBox=(2219, 2112, 2368, 2527)
Misplaced Product: Class=class fanta, Row=5, Product=15, BBox=(2372, 2109, 2541, 2527)
Misplaced Product: Class=class fanta, Row=5, Product=16, BBox=(2547, 2109, 2712, 2524)
Misplaced Product: Class=class fanta, Row=5, Product=17, BBox=(2716, 2107, 2869, 2523)
Misplaced Product: Class=class sprite, Row=5, Product=21, BBox=(3373, 2106, 3456, 2511)
Misplaced Product: Class=class coke, Row=6, Product=1,